In [80]:
import pandas as pd
import glob
from datetime import date, timedelta
import numpy as np
from datetime import datetime
import pathlib
from pathlib import Path
from collections import OrderedDict
import polars as pl
import fastexcel
import os
import time

In [81]:
def convert_to_datetime(struct_time):
    """Convert struct_time to datetime object."""
    return datetime(*struct_time[:6])

def input_data(folder_path, sheet_name=None):
    file_paths = glob.glob(f"{folder_path}/*.xlsx") + glob.glob(f"{folder_path}/*.csv")
    df_list = []

    for file in file_paths:
        export_time = os.path.getmtime(file)
        export_time_datetime = convert_to_datetime(time.localtime(export_time))

        if file.endswith('.xlsx'):
            df = pl.read_excel(file, sheet_name=sheet_name)
        elif file.endswith('.csv'):
            try:
                df = pl.read_csv(file, encoding="utf-8", infer_schema_length=5000)
            except:
                df = pl.read_csv(file, encoding="ISO-8859-1", ignore_errors=True, infer_schema_length=5000)

        df = df.with_columns([
            pl.col(col).cast(pl.String)
            for col in df.columns
        ])

        df = df.with_columns([
            pl.lit(os.path.basename(file)).alias('File Name'),
            pl.lit(export_time_datetime).alias('Export Time')
        ])

        df_list.append(df)

    if df_list:
        merged_df = pl.concat(df_list, how='vertical')
    else:
        merged_df = pl.DataFrame()

    return merged_df

def input_data_parquet(folder_path):
    file_paths = glob.glob(f"{folder_path}/*.parquet")
    df_list = []

    for file in file_paths:
        export_time = os.path.getmtime(file)
        export_time_datetime = convert_to_datetime(time.localtime(export_time))

        df = pl.read_parquet(file)   # Nhanh, tiết kiệm RAM

        df = df.with_columns([
            pl.lit(os.path.basename(file)).alias('File Name'),
            pl.lit(export_time_datetime).alias('Export Time')
        ])
        df_list.append(df)

    if df_list:
        merged_df = pl.concat(df_list, how='vertical')
    else:
        merged_df = pl.DataFrame()

    return merged_df

def csv_to_parquet(input_folder, output_folder=None):
    import polars as pl
    import glob, os

    if output_folder is None:
        output_folder = input_folder
    os.makedirs(output_folder, exist_ok=True)
    csv_files = glob.glob(os.path.join(input_folder, "*.csv"))
    if not csv_files:
        return
    for csv_file in csv_files:
        try:
            df = pl.read_csv(
                csv_file,
                infer_schema_length=10000,  # mở rộng để infer tốt hơn
                schema_overrides={
                    "Promoter Score (Calc)": pl.Float64,
                    "Detractor Score (Calc)": pl.Float64,
                    # Thêm các cột tương tự nếu cần!
                }
            )
            parquet_file = os.path.splitext(os.path.basename(csv_file))[0] + ".parquet"
            parquet_path = os.path.join(output_folder, parquet_file)
            df.write_parquet(parquet_path)
        except Exception as e:
            print(f"Không convert được file: {csv_file} - Lỗi: {e}")
        
today_temp = datetime.today().date()
today = today_temp.strftime('%b_%d_%Y')

In [82]:
first_glob = os.path.expanduser("~").replace("\\", "/")

folder_paths = {
    "input_performance":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/NEW_LOOK_EXCEL_EN/new_look_excel_data',
    "output_en_performance_global":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/Global Report/performance_en',
    "input_survey":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/SURVEY_EN',
    "input_t3":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/T3_EN',
    "input_afcr":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/FCR',
    "input_delayed_closure":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/DELAYED_CLOSURE',
    "mapping_file": f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/AQG_summarized.xlsx',
    "global_hc": f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Global_HC.parquet',
    "input_parquet_performance":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW//RAW/PARQUET',
    "input_csv_re_direct":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/RE-DIRECT'
}

print("--- FULL FOLDER PATHS LIST ---")
for key, path in folder_paths.items():
    print(f"{key}: {path}")

print("-" * 60)

--- FULL FOLDER PATHS LIST ---
input_performance: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/NEW_LOOK_EXCEL_EN/new_look_excel_data
output_en_performance_global: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/Global Report/performance_en
input_survey: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/SURVEY_EN
input_t3: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/T3_EN
input_afcr: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/FCR
input_delayed_closure: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/DELAYED_CLOSURE
mapping_file: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/AQG_summarized.xlsx
global_hc: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Global_HC.parquet
input_parquet_performance: C:/Use

In [83]:
csv_to_parquet(folder_paths["input_performance"], folder_paths["input_parquet_performance"])

In [84]:
SURVEY_INPUT = input_data(folder_paths["input_survey"])

survey_added_ir = SURVEY_INPUT.with_columns([
    pl.when(
        (pl.col("Question Category") == "Resolution") & (pl.col("Answer") == "Yes")
    ).then(1)
    .when(
        (pl.col("Question Category") == "Resolution") & (pl.col("Answer") == "No")
    ).then(0)
    .otherwise(0)
    .alias("_ir")
])

survey_ae = (
    SURVEY_INPUT
    .filter(pl.col("Question Category") == "agentExperience")
    .with_columns(
        pl.col("Answer")
          .cast(pl.Int64, strict=False)
          .alias("_ae")
    )
)

survey_verbatim = (
    SURVEY_INPUT
    .filter(pl.col("Question Category") == "Feedback")
    .with_columns([
        pl.col("Answer").str.replace_all(r"[\r\n\t]+", " ").alias("_verbatim"),
        pl.concat_str([
            pl.col("Agent Email ID").cast(pl.String).fill_null(""),
            pl.col("Conversation Id").cast(pl.String).fill_null(""),
            pl.col("Joined Time").str.to_datetime(strict=False).dt.strftime("%y%m%d%H%M%S")
        ], separator="_").alias("_conver_unique"),
    ])
)

metrics = ["d_happy_response", "d_surprised_response", "e_response", "t_response", "u_response"]

survey_duet = (
    SURVEY_INPUT
    .filter(pl.col("Question Category").is_in(metrics))
    .with_columns([
        pl.col("Question Category").alias("metric"),
        pl.col("Answer").cast(pl.Float64, strict=False).alias("value")
    ])
    .group_by(["Conversation Id", "metric"])
    .agg(pl.max("value").alias("value")) 
    .pivot(values="value", index="Conversation Id", columns="metric")
)

# --- Excel-equivalent calculations using column names (force Float64) ---
survey_duet = survey_duet.with_columns([
    pl.when(pl.any_horizontal([pl.col("d_happy_response").is_null(),
                               pl.col("d_surprised_response").is_null()]))
      .then(pl.lit(None, dtype=pl.Float64))
      .otherwise(pl.when((pl.col("d_happy_response") >= 4) & (pl.col("d_surprised_response") >= 3))
                   .then(100.0).otherwise(0.0))
      .alias("delight"),

    pl.when(pl.col("u_response").is_null()).then(pl.lit(None, dtype=pl.Float64))
      .otherwise(((pl.col("u_response") - 1.0) / 6.0) * 100.0).alias("usability"),

    pl.when(pl.col("e_response").is_null()).then(pl.lit(None, dtype=pl.Float64))
      .otherwise(((pl.col("e_response") - 1.0) / 6.0) * 100.0).alias("ease"),

    pl.when(pl.col("t_response").is_null()).then(pl.lit(None, dtype=pl.Float64))
      .otherwise(((pl.col("t_response") - 1.0) / 6.0) * 100.0).alias("trust"),

    pl.when(pl.any_horizontal([pl.col("d_happy_response").is_null(),
                               pl.col("d_surprised_response").is_null(),
                               pl.col("u_response").is_null(),
                               pl.col("e_response").is_null(),
                               pl.col("t_response").is_null()]))
      .then(pl.lit(None, dtype=pl.Float64))
      .otherwise(pl.when((pl.col("d_happy_response") >= 4) &
                         (pl.col("d_surprised_response") >= 3) &
                         ((pl.col("u_response") + pl.col("e_response")) >= 13) &
                         (pl.col("e_response") >= 6) &
                         (pl.col("t_response") >= 6))
                   .then(100.0).otherwise(0.0))
      .alias("DUET"),
])

# Round computed columns
survey_duet = survey_duet.with_columns([
    pl.col("delight").round(0),
    pl.col("usability").round(0),
    pl.col("ease").round(0),
    pl.col("trust").round(0),
    pl.col("DUET").round(0),
])

# Keep ALL required columns (raw + computed)
final_cols = ["Conversation Id"] + metrics + ["delight", "usability", "ease", "trust", "DUET"]
survey_duet = survey_duet.select([c for c in final_cols if c in survey_duet.columns])
survey_ir = (survey_added_ir.select(["Conversation Id", "_ir"]).filter(pl.col("_ir") == 1).unique())
survey_ae = (survey_ae.select(["Conversation Id", "_ae"]).filter(pl.col("_ae") > 0).unique())
verbatim = (survey_verbatim.select(["_conver_unique", "_verbatim"])).unique()

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_1328\1549389880.py:48: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  .pivot(values="value", index="Conversation Id", columns="metric")


In [85]:
T3_INPUT = input_data(folder_paths["input_t3"])

t3_added_col_t3 = T3_INPUT.with_columns([
    pl.when(
        (pl.col("Agent Queue Group Name").str.contains("T3", literal=True)) &
        (pl.col("Transferred (Yes/No)") == "Yes")
    ).then(1).otherwise(0).alias("T3")])

t3_final = (t3_added_col_t3.select(["Conversation Id", "T3"]).filter(pl.col("T3") == 1).unique())

In [86]:
def process_afcr_folder(folder_path: str) -> pl.DataFrame:
    import datetime
    all_dataframes = []
    for file_name in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file_name)
        if not os.path.isfile(file_path):
            continue
        timestamp = os.path.getmtime(file_path)
        export_time = datetime.datetime.fromtimestamp(timestamp).strftime('%Y-%m-%d %H:%M:%S')
        if file_name.endswith(('.xlsx', '.xls')):
            df = pl.read_excel(file_path, engine='calamine')
            df = df.with_columns(pl.col("Itinerary").cast(pl.String))
        elif file_name.endswith('.csv'):
            df = pl.read_csv(
                file_path, 
                encoding="utf-8", 
                schema_overrides={"Itinerary": pl.String},
                infer_schema_length=10000,
                ignore_errors=True
            )
        else:
            continue
        df = df.with_columns([
            pl.col("Handle Time").cast(pl.Float64, strict=False),
            pl.col("Duet").cast(pl.Float64, strict=False),
            pl.col("Passed Sessions").cast(pl.Int64, strict=False),
            pl.col("Failed Sessions").cast(pl.Int64, strict=False)
        ])
        
        df = df.with_columns([
            pl.lit(file_name).alias('File Name'),
            pl.lit(export_time).alias('Export Time')
        ])
        all_dataframes.append(df)
        
    if not all_dataframes:
        return pl.DataFrame()
    return pl.concat(all_dataframes, how="diagonal_relaxed")

afcr_input = process_afcr_folder(folder_paths["input_afcr"])

afcr_input = (
    afcr_input
    .filter(pl.col("Vendor Partner Location") == "Concentrix (Ho Chi Minh City)")
    .select([
        pl.col("Agent Email Address").alias("Agent Email ID"),
        pl.col("Conversation Id"),
        pl.col("Passed Sessions"),
        pl.col("Failed Sessions"),
        pl.when(pl.col("Passed Sessions").is_in([0, 1]))
        .then(1)
        .otherwise(0)
        .alias("Total Sessions")
    ])
    .unique()
)

In [87]:
DELAYED_CLOSURE_INPUT = (
    input_data(folder_paths["input_delayed_closure"])
    .filter(pl.col("Agent Vendor Location") == "Concentrix (Ho Chi Minh City)")
    .unique(subset=["Agent Email ID", "Conversation Id", "Joined Time"], keep='last'))
delayed_closure = (
    DELAYED_CLOSURE_INPUT
    .select([
        "Agent Email ID", "Conversation Id", "Joined Time", "Traveler Unresponsive Time", "Response Time",
        "Agent Disconnect (Yes / No)", "Ghost (Yes / No)", "Requeued (Yes / No)"
    ])
    .with_columns([
        pl.col("Joined Time").str.to_datetime().cast(pl.Datetime),
        pl.col("Traveler Unresponsive Time").cast(pl.Float64),
        pl.col("Response Time").cast(pl.Float64),
        (pl.col("Traveler Unresponsive Time").cast(pl.Float64) - 240)
            .clip(0, None)
            .alias("Exceed Time"),
        ((pl.col("Traveler Unresponsive Time").cast(pl.Float64) - 240) > 0)
            .cast(pl.Int8)
            .alias("Exceed Chat"),
        (pl.col("Agent Disconnect (Yes / No)") == "Yes").cast(pl.Int8).alias("Agent Disconnect"),
        (pl.col("Ghost (Yes / No)") == "Yes").cast(pl.Int8).alias("Ghost"),
        (pl.col("Requeued (Yes / No)") == "Yes").cast(pl.Int8).alias("Requeued"),
        (pl.col("Response Time").cast(pl.Float64).is_null() | (pl.col("Response Time").cast(pl.Float64) == 0))
            .cast(pl.Int8)
            .alias("Traveler Unresponsive")
    ])
    .drop([
        "Agent Disconnect (Yes / No)", "Ghost (Yes / No)", "Requeued (Yes / No)",
        "Traveler Unresponsive Time", "Response Time"
    ])
    .group_by(["Agent Email ID", "Conversation Id", "Joined Time"])
    .agg([
        pl.sum("Exceed Time").alias("Exceed Time"),
        pl.sum("Exceed Chat").alias("Exceed Chat"),
        pl.sum("Agent Disconnect").alias("Agent Disconnect"),
        pl.sum("Ghost").alias("Ghost"),
        pl.sum("Requeued").alias("Requeued"),
        pl.sum("Traveler Unresponsive").alias("Traveler Unresponsive")
    ])
)

delayed_closure = delayed_closure.with_columns(
    pl.when(pl.col("Exceed Time") <= 0).then(pl.lit(None, dtype=pl.Utf8))
      .when((pl.col("Exceed Time")/60) <= 1).then(pl.lit("01 Mins"))
      .when((pl.col("Exceed Time")/60) <= 2).then(pl.lit("02 Mins"))
      .when((pl.col("Exceed Time")/60) <= 3).then(pl.lit("03 Mins"))
      .when((pl.col("Exceed Time")/60) <= 4).then(pl.lit("04 Mins"))
      .when((pl.col("Exceed Time")/60) <= 5).then(pl.lit("05 Mins"))
      .when((pl.col("Exceed Time")/60) <= 10).then(pl.lit("05-10 Mins"))
      .when((pl.col("Exceed Time")/60) <= 15).then(pl.lit("10-15 Mins"))
      .when((pl.col("Exceed Time")/60) <= 30).then(pl.lit("15-30 Mins"))
      .otherwise(pl.lit("30+ Min"))
      .alias("Exceed Bucket")
)
print(delayed_closure.height)

51475


In [88]:
RE_DIRECT_INPUT = (
    input_data(folder_paths["input_csv_re_direct"])
    .with_columns(
        PeopleID_ConversationID_KEY=pl.concat_str(
            [pl.col("Agent People Id").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)],
            separator="_",
        ),
        Re_Direct=pl.lit(1),
        **{
            "Re-Direct Text": (
                pl.col("Text").cast(pl.Utf8).fill_null("")
                .str.replace_all(r"(\r\n|\r|\n)+", " | ")
                .str.replace_all(r"[\-•\u2022\u25CF\u25E6\u2043\u2219\u00B7\u2013\u2014]+", "")
                .str.replace_all(r"\s{2,}", " ")
                .str.strip_chars()
            )
        }
    )
    .select(["PeopleID_ConversationID_KEY", "Re_Direct", "Re-Direct Text"])
    .unique(maintain_order=True)
)

In [89]:
PERFORMANCE_INPUT = input_data(folder_paths["input_performance"])

try: 
    if PERFORMANCE_INPUT.columns[0] == "":
        PERFORMANCE_INPUT = PERFORMANCE_INPUT.drop(PERFORMANCE_INPUT.columns[0])
except: pass

print(PERFORMANCE_INPUT.columns)
existing_cols = set(PERFORMANCE_INPUT.columns)

columns_to_cast = {
    "Joined Time": pl.Datetime,
    "Handle Time (Sum)": pl.Float64,
    "Handle (Count)": pl.Int64,
    "Hold Time (Sum)": pl.Float64,
    "Talk Time (Sum)": pl.Float64,
    "Wrap Up Time (Sum)": pl.Float64,
    "Survey Submitted (Count)":pl.Float64,
    "Promoter Score (Calc)": pl.Float64,
    "Detractor Score (Calc)": pl.Float64,
    "Neutral Score (Calc)": pl.Float64,
    "Has Followup Time": pl.Float64,
    "Response Time (Sum)": pl.Float64,
    "Response Time (Avg)": pl.Float64,
    "Concurrency": pl.Float64,
    "Survey Offered (Count)":pl.Int64,
}

datetime_cols = ["Started Time", "Joined Time", "Submitted Time", "Left Time"]

casts = []

for col, dtype in columns_to_cast.items():
    if col in existing_cols:
        if col == "Submitted Date":
            casts.append(pl.col(col).str.strptime(pl.Date, "%m/%d/%Y", strict=False))
        elif col in datetime_cols:
            casts.append(
                pl.when(pl.col(col).str.contains(r"^\d{4}-\d{2}-\d{2}"))  # ISO: YYYY-MM-DD
                  .then(pl.col(col).str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S", strict=False))
                .when(pl.col(col).str.contains(r"^\d{1,2}/\d{1,2}/\d{4}"))  # US: M/D/YYYY
                  .then(pl.col(col).str.strptime(pl.Datetime, "%m/%d/%Y %H:%M", strict=False))
                .otherwise(None)
                .alias(col)
            )
        elif dtype == pl.Float64:
            casts.append(
                pl.col(col)
                .cast(pl.Utf8)
                .str.replace_all(",", "")
                .cast(pl.Float64, strict=False)
                .alias(col)
            )
        else:
            casts.append(pl.col(col).cast(dtype).alias(col))

PERFORMANCE_CHANGED_TYPE = PERFORMANCE_INPUT.with_columns(casts)
PERFORMANCE_ADDED_CCR72 = PERFORMANCE_CHANGED_TYPE.with_columns([
    pl.when(pl.col("Has Followup Time").cast(pl.Float64) <= 4320)
      .then(1.0)
      .otherwise(0.0)
      .alias("CCR72")
])

PERFORMANCE_NEXT_STEP = PERFORMANCE_ADDED_CCR72.with_columns([
    pl.col("Joined Time").dt.date().alias("Joined Date")
])

PERFORMANCE_NEXT_STEP = PERFORMANCE_NEXT_STEP.with_columns([
    (pl.col("Joined Time") + pl.duration(hours=14)).alias("Join Time (VNT)"),
    (pl.col("Joined Time") + pl.duration(hours=14)).dt.date().alias("Join Date (VNT)")
])

GLOBAL_HC = pl.read_parquet(folder_paths["global_hc"]).unique(subset=["SSO ID"], keep="last")

merged_global_hc = PERFORMANCE_NEXT_STEP.join(GLOBAL_HC[['SSO ID', "Name", "Supervisor", "Production Start date", "Agent/Non Agent"]], left_on=["Agent Email ID"], right_on=['SSO ID'], how='left')
mapping = pl.read_excel(folder_paths["mapping_file"], sheet_name="queue_name_mapping")
mapping_harmony = pl.read_excel(folder_paths["mapping_file"], sheet_name="harmony_agents")
mapping_lc_scheme = pl.read_excel(folder_paths["mapping_file"], sheet_name="kpi")
mapping_bq_list = pl.read_excel(folder_paths["mapping_file"],sheet_name="bq_list")
mapping_bq_list = (mapping_bq_list.filter(pl.col("Sub") == "en").select([pl.col("Agent Email ID").cast(pl.String),pl.col("Quartile Flag").cast(pl.String)]))

performance_cleaned = merged_global_hc.join(mapping, on="Agent Queue Group Name",how='left')
performance_cleaned = (performance_cleaned.join(mapping_harmony, on="Agent Email ID", how="left").with_columns(pl.col("Harmony Flag").fill_null("Non Harmony")))
performance_cleaned = performance_cleaned.join(mapping_bq_list,on="Agent Email ID",how="left")

offered = (performance_cleaned.group_by(["Conversation Id", "Survey Offered (Count)"]).agg(pl.count().alias("Count"))
    .with_columns(
        (pl.col("Survey Offered (Count)") / pl.col("Count")).alias("Survey Offered")
    )
)

performance_cleaned = (performance_cleaned.join(offered, on="Conversation Id",how='left').with_columns(pl.col("Survey Offered").fill_null(0)))
LOB_MAPPING_SHOWED = mapping.filter(pl.col("Agent Queue Group Name").str.contains(r"[dD][aA]"))
lc_mapping = pl.read_excel(folder_paths["mapping_file"], sheet_name="kpi")

['Joined Date', 'Agent People Id', 'Conversation Id', 'Agent Email ID', 'Agent Queue Group Name', 'Latest VA Intent', 'Agent Business Location', 'Initiated Outbound (Yes / No)', 'Latest VA Product', 'Response Count', 'Response Time', 'Customer Loyalty Status', 'Business Segment Name', 'Partner Name', 'Language', 'Effective Disconnected By', 'Joined Time', 'Left Time', 'Entry Point App', 'Device Category', 'Device Os', 'Browser', 'Has Followup Time', 'Queue Name', 'Agent Manager Name', 'Agent Name', 'Handle (Count)', 'Handle Time (Sum)', 'Wrap Up Time (Sum)', 'Hold Time (Sum)', 'Talk Time (Sum)', 'Follow Up (Count)', 'Survey Submitted (Count)', 'Promoter Score (Calc)', 'Detractor Score (Calc)', 'Assigned Agent Time (Sum)', 'Sum of Chat Agent First Response Time', 'Survey Offered (Count)', 'File Name', 'Export Time']


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_1328\8450888.py:87: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  offered = (performance_cleaned.group_by(["Conversation Id", "Survey Offered (Count)"]).agg(pl.count().alias("Count"))


In [90]:
performance_processed = performance_cleaned.with_columns([
    (1 - pl.col("Promoter Score (Calc)") - pl.col("Detractor Score (Calc)"))
        .alias("Neutral Score (Calc)")
])

performance_processed = performance_processed.with_columns([
    (pl.col("Promoter Score (Calc)") * pl.col("Survey Submitted (Count)")).round(0).alias("_promoter"),
    (pl.col("Detractor Score (Calc)") * pl.col("Survey Submitted (Count)")).round(0).alias("_detractor"),
    (pl.col("Neutral Score (Calc)") * pl.col("Survey Submitted (Count)")).round(0).alias("_neutral")
])

performance_processed = performance_processed.with_columns([

    pl.when(pl.col("Initiated Outbound (Yes / No)") == "Yes").then(1).otherwise(0).alias("_aob"),

    pl.col("Survey Submitted (Count)").alias("_survey"),
    pl.col("CCR72").cast(pl.Float64).fill_null(0.0).alias("_fup_72"),
    (1.0 - pl.col("CCR72").cast(pl.Float64).fill_null(0.0)).alias("_rr"),

    pl.col("Joined Time").dt.date().alias("_PST.Date"),
    pl.col("Joined Time").dt.strftime("%y_%m").alias("_PST.Month"),
    pl.col("Joined Time").dt.strftime("%G_%V").str.slice(2).alias("_PST.WeekNo"),
    (pl.col("Joined Time") - pl.col("Joined Time").dt.weekday() * pl.duration(days=1)).dt.strftime("WB_%y/%m/%d").alias("_PST.Week"),
    pl.col("Joined Time").dt.year().alias("_PST.Year"),

    pl.concat_str([
        pl.col("Agent Email ID").cast(pl.Utf8).fill_null(""),
        pl.col("Conversation Id").cast(pl.Utf8).fill_null(""),
        pl.col("Joined Time").dt.strftime("%y%m%d%H%M%S")
    ], separator="_").alias("_conver_unique"),

    pl.concat_str([
        pl.col("Agent Email ID").cast(pl.Utf8).fill_null(""),
        pl.col("Conversation Id").cast(pl.Utf8).fill_null("")], 
        separator="_").alias("_key"),

    pl.col("Survey Offered").alias("_offer"),
    pl.when(pl.col("Joined Time") <= pl.datetime(2025, 7, 7)).then(pl.lit("Pre")).otherwise(pl.lit("Post")).alias("Pre/Post")
])

performance_processed = performance_processed.with_columns([
    (
        (pl.col("_PST.Date").cast(pl.Date) - pl.col("Production Start date").cast(pl.Date))
        / pl.duration(days=1)
    ).cast(pl.Int32).alias("AON_Days")
])
performance_processed = performance_processed.with_columns([
    pl.when(
        pl.col("AON_Days").is_null() & (
            (pl.col("Agent/Non Agent") == "Agent") | (pl.col("Agent/Non Agent") == "ID Deleted")
        )
    )
    .then(pl.lit("Nesting"))
    .when(pl.col("AON_Days") > 180)
      .then(pl.lit("> 180 Days"))
    .when(pl.col("AON_Days") >= 91)
      .then(pl.lit("91 - 180"))
    .when(pl.col("AON_Days") >= 61)
      .then(pl.lit("61 - 90"))
    .when(pl.col("AON_Days") >= 31)
      .then(pl.lit("31 - 60"))
    .when(pl.col("AON_Days") >= 0)
      .then(pl.lit("00 - 30"))
    .otherwise(None)
    .alias("AON Status")
])
performance_processed = performance_processed.join_asof(
    mapping_lc_scheme,
    left_on="_PST.Date",
    right_on="Effective Date",
    by="LOB",
    strategy="backward"
)
performance_processed = performance_processed.with_columns([
    (pl.col("Handle Time (Sum)") >= pl.col("Threshole_LC")).cast(pl.Int8).alias("_lc")
])

performance_processed = performance_processed.with_columns([
    (pl.col("Handle Time (Sum)") < 240).cast(pl.Int8).alias("Short Chat")
])

print(performance_processed.select(["_PST.Month","Threshole_LC", "Handle Time (Sum)", "_lc"]).filter(pl.col("_lc") == 1))

performance_processed = performance_processed.with_columns([
    pl.concat_str([pl.col("Agent Email ID"), pl.col("_PST.Date").dt.strftime("%y%m%d")]).alias("KEY")
])

performance_processed = performance_processed.with_columns([
    pl.concat_str([pl.col("Agent Email ID").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)],separator="_").alias("EmailID_ConversationID_KEY"),
    pl.concat_str([pl.col("Agent People Id").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)],separator="_").alias("PeopleID_ConversationID_KEY"),
])

performance_processed = performance_processed.with_columns([
    (
        pl.when(pl.col("_promoter") > 0)
          .then(pl.lit("promoter, "))
          .otherwise(pl.lit(""))
      +
        pl.when(pl.col("_detractor") > 0)
          .then(pl.lit("detractor, "))
          .otherwise(pl.lit(""))
      +
        pl.when(pl.col("_neutral") > 0)
          .then(pl.lit("neutral"))
          .otherwise(pl.lit(""))
    ).str.strip_chars(", ").alias("_nps_type")
])

def get_30min_interval(dt):
    if dt is None:
        return None
    hour = dt.hour
    minute = dt.minute
    if minute < 30:
        start = datetime(dt.year, dt.month, dt.day, hour, 0)
        end = start + timedelta(minutes=29)
    else:
        start = datetime(dt.year, dt.month, dt.day, hour, 30)
        end = start + timedelta(minutes=29)
    return f"{start.strftime('%H:%M')}-{end.strftime('%H:%M')}"

def get_period(dt):
    if dt is None:
        return None
    hour = dt.hour
    if hour >= 18:
        return 'Night'
    elif hour >= 12:
        return 'Mid'
    else:
        return 'Morning'

print(performance_processed.columns)

performance_processed = performance_processed.with_columns([
    pl.col('Joined Time').map_elements(get_30min_interval, return_dtype=pl.Utf8).alias('_PST.Interval')
])

performance_merged_survey_t3 = (
    performance_processed
    .join(survey_ir, on = "Conversation Id", how="left")
    .join(survey_ae, on = "Conversation Id", how="left")
    .join(survey_duet, on = "Conversation Id", how="left")
    .join(t3_final, on = "Conversation Id", how="left")
    .join(delayed_closure, on = ["Agent Email ID", "Conversation Id", "Joined Time"], how="left")
    .join(RE_DIRECT_INPUT, on = "PeopleID_ConversationID_KEY", how="left")
    .join(verbatim, on = ["_conver_unique"], how="left")
    .join(afcr_input, on = ["Agent Email ID", "Conversation Id"], how="left")
)

print(performance_merged_survey_t3.columns)
print(performance_merged_survey_t3.select(["_PST.Date", "Production Start date", "AON_Days"]).head())

selected_columns = [
    "_PST.Date","_PST.Year","_PST.WeekNo","_PST.Week","_PST.Month", "_PST.Interval",
    "Agent People Id","Agent Email ID","Conversation Id","Joined Time","Left Time",
    "Agent Queue Group Name","Supervisor","Agent Business Location","AON Status","Name","Language","LOB",
    "Latest VA Intent","Latest VA Product","Business Segment Name","Partner Name",
    "Handle (Count)","Handle Time (Sum)","Response Count","Response Time","Hold Time (Sum)",
    "Wrap Up Time (Sum)","Talk Time (Sum)","_aob","CCR72","_offer","_survey","_promoter",
    "_detractor","_neutral","_nps_type","_lc","Customer Loyalty Status","Effective Disconnected By",
    "Sum of Chat Agent First Response Time","T3","_ir", "_verbatim", "Agent/Non Agent","_ae","Harmony Flag",
    "Pre/Post","_conver_unique","Re_Direct","Exceed Time","Exceed Chat","Agent Disconnect", 
    "Ghost","Requeued","Exceed Bucket","d_happy_response","d_surprised_response",
    "e_response","t_response","u_response",'delight', 'usability', 'ease', 'trust', 'DUET', 
    "Traveler Unresponsive", "Short Chat",  "Passed Sessions", "Failed Sessions", "Total Sessions", 'Quartile Flag'
]


missing_cols = [col for col in selected_columns if col not in performance_merged_survey_t3.columns]
print("Các cột bị thiếu:", missing_cols)

performance_filtered = performance_merged_survey_t3.select(selected_columns)
performance_filtered = performance_filtered.unique()
performance_filtered = performance_filtered.sort(
    by=["Conversation Id", "Agent Email ID", "Joined Time"],
    descending=[False, False, True]
).with_columns(
    (pl.col("Joined Time").cum_count().over(["Conversation Id", "Agent Email ID"]) > 1)
    .cast(pl.Int8)
    .alias("Duplicate_Flag")
)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_1328\1257070165.py:67: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  performance_processed = performance_processed.join_asof(


shape: (11_077, 4)
┌────────────┬──────────────┬───────────────────┬─────┐
│ _PST.Month ┆ Threshole_LC ┆ Handle Time (Sum) ┆ _lc │
│ ---        ┆ ---          ┆ ---               ┆ --- │
│ str        ┆ i64          ┆ f64               ┆ i8  │
╞════════════╪══════════════╪═══════════════════╪═════╡
│ 26_04      ┆ 2700         ┆ 6334.678          ┆ 1   │
│ 26_04      ┆ 1200         ┆ 2341.0            ┆ 1   │
│ 26_04      ┆ 2400         ┆ 2663.0            ┆ 1   │
│ 26_04      ┆ 1200         ┆ 1527.0            ┆ 1   │
│ 26_04      ┆ 2100         ┆ 2241.037          ┆ 1   │
│ …          ┆ …            ┆ …                 ┆ …   │
│ 26_04      ┆ 1200         ┆ 2743.0            ┆ 1   │
│ 26_04      ┆ 2700         ┆ 2934.924          ┆ 1   │
│ 26_04      ┆ 1500         ┆ 1797.968          ┆ 1   │
│ 26_04      ┆ 1200         ┆ 1212.0            ┆ 1   │
│ 26_04      ┆ 2100         ┆ 3160.395          ┆ 1   │
└────────────┴──────────────┴───────────────────┴─────┘
['Joined Date', 'Agent People

In [91]:
# 1) Read the removal list from Excel
removed_id = pl.read_excel(folder_paths["mapping_file"], sheet_name="id_removed")

# 2) Helper to normalize Conversation Id (cast -> drop quotes -> trim spaces)
def clean_id(expr: pl.Expr) -> pl.Expr:
    return (
        expr.cast(pl.Utf8)
            .str.replace_all(r'["“”]', "")      # remove any quote characters
            .str.replace_all(r'^\s+|\s+$', "")  # trim leading/trailing whitespace (regex)
    )

# Validate required column
if "Conversation Id" not in removed_id.columns:
    raise ValueError("Sheet 'id_removed' must contain a 'Conversation Id' column.")

# Canonical removal list: unique, non-null, non-empty
removed_id_norm = (
    removed_id
    .select(clean_id(pl.col("Conversation Id")).alias("Conversation Id"))
    .drop_nulls()
    .filter(pl.col("Conversation Id") != "")
    .unique()
)

# 3) Instead of removing rows, add a flag column "IDs Removed" (Yes/No)
performance_filtered = (
    performance_filtered
    .with_columns(clean_id(pl.col("Conversation Id")).alias("__conv_norm"))
    .join(
        removed_id_norm
            .select(pl.col("Conversation Id").alias("__conv_norm"))
            .with_columns(pl.lit(1).alias("__rm")),
        on="__conv_norm",
        how="left",
    )
    .with_columns(pl.col("__rm").fill_null(0).alias("IDs Removed"))
    .drop(["__conv_norm", "__rm"])
)

# (Optional) thống kê nhanh
flagged = performance_filtered.select(pl.col("IDs Removed").sum().alias("flagged")).item()
total = performance_filtered.height
print(f"Flagged (IDs Removed=1): {flagged} / {total} rows")

Flagged (IDs Removed=1): 0 / 246799 rows


In [92]:
# region EXPORT
performance_unique = performance_filtered.unique()
performance_all_site = performance_unique

output_dir = folder_paths["output_en_performance_global"]

for (month_value,), group in performance_all_site.group_by(['_PST.Month'], maintain_order=True):
    base_name = str(month_value)
    group.write_csv(os.path.join(output_dir, f"{base_name}.csv"))
    group.write_parquet(os.path.join(output_dir, f"{base_name}_performance_en_vn.parquet"))

parquet_files = glob.glob(os.path.join(output_dir, "*_performance_en_vn.parquet"))
out_path_total = os.path.join(output_dir, "all_months_performance_en_global.parquet")
lazy_dfs = [pl.scan_parquet(file) for file in parquet_files]
(
    pl.concat(lazy_dfs, how="vertical_relaxed")
    .collect()
    .write_parquet(out_path_total)
)# endregion

In [93]:
print(performance_all_site['_PST.Month'].unique())

shape: (1,)
Series: '_PST.Month' [str]
[
	"26_04"
]


In [94]:
print(performance_all_site.columns)

['_PST.Date', '_PST.Year', '_PST.WeekNo', '_PST.Week', '_PST.Month', '_PST.Interval', 'Agent People Id', 'Agent Email ID', 'Conversation Id', 'Joined Time', 'Left Time', 'Agent Queue Group Name', 'Supervisor', 'Agent Business Location', 'AON Status', 'Name', 'Language', 'LOB', 'Latest VA Intent', 'Latest VA Product', 'Business Segment Name', 'Partner Name', 'Handle (Count)', 'Handle Time (Sum)', 'Response Count', 'Response Time', 'Hold Time (Sum)', 'Wrap Up Time (Sum)', 'Talk Time (Sum)', '_aob', 'CCR72', '_offer', '_survey', '_promoter', '_detractor', '_neutral', '_nps_type', '_lc', 'Customer Loyalty Status', 'Effective Disconnected By', 'Sum of Chat Agent First Response Time', 'T3', '_ir', '_verbatim', 'Agent/Non Agent', '_ae', 'Harmony Flag', 'Pre/Post', '_conver_unique', 'Re_Direct', 'Exceed Time', 'Exceed Chat', 'Agent Disconnect', 'Ghost', 'Requeued', 'Exceed Bucket', 'd_happy_response', 'd_surprised_response', 'e_response', 't_response', 'u_response', 'delight', 'usability', 'ea

In [95]:
quartile_df = (
    performance_all_site
    .group_by(['_PST.Month', '_PST.WeekNo', 'Agent Business Location', 'Agent Email ID', 'Agent Queue Group Name', 'Latest VA Product'])
    .agg([
        pl.col('_promoter').sum().alias('Promoter'),
        pl.col('_detractor').sum().alias('Detractor'),
        pl.col('_survey').sum().alias('Survey')
    ])
    .with_columns([
        pl.when(pl.col('Survey').is_null() | (pl.col('Survey') == 0))
        .then(None)
        .otherwise(((pl.col('Promoter') - pl.col('Detractor')) / pl.col('Survey') * 100))
        .alias('NPS')
    ])
)
 
quartile_df = (
    quartile_df
    .with_columns([
        pl.when(pl.col('NPS').is_null()).then(None)
        .when(pl.col('NPS') >= 75).then(pl.lit('Quartile 1'))
        .when(pl.col('NPS') >= 50).then(pl.lit('Quartile 2'))
        .when(pl.col('NPS') >= 25).then(pl.lit('Quartile 3'))
        .when(pl.col('NPS') < 25).then(pl.lit('Quartile 4'))
        .otherwise(None)
        .alias('Quartile')
    ])
)
 
quartile_df.write_excel('quartile2.xlsx')